<a href="https://colab.research.google.com/github/SehrishbAsghar/FlyRank_ML_Internship_Sehrish/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [7]:
##  Task type: Scoring (a form of regression/ranking) not classification or clustering.

# I am not sorting pages into fixed categories (classification) or grouping similar pages together (clustering).
# I am assigning each page a continuous opportunity score that reflects how much its CTR underperforms what is expected for its position tier and content type
# then ranking pages by that score. This is a scoring/ranking task because the output is an ordered list a reviewer works down, not a label.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [8]:
# I don't have a direct "opportunity" label in the data, no one has tagged which pages are truly underperforming. So I need a proxy target.
## Proxy target:
# The gap between a page's actual CTR and the expected CTR for its position_tier + content_type group, adjusted for how much sample confidence that group has.
# A large negative gap = underperforming for its position/type.
# A large positive gap = overperforming. This proxy is imperfect
# it's a statistical comparison, not a verified label but it is the best signal available without manual review data.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [9]:
# Since this feeds a reviewer working down a ranked list, Precision @ K is the right metric
# I am deliberately not using a metric like raw correlation or R-squared, because the real-world use case isn't "explain all the variance"
# it is "give a reviewer a short, trustworthy list to act on.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
!git clone https://github.com/SehrishbAsghar/FlyRank_ML_Internship_Sehrish.git
%cd FlyRank_ML_Internship_Sehrish

Cloning into 'FlyRank_ML_Internship_Sehrish'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 103 (delta 22), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 1.87 MiB | 4.96 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/FlyRank_ML_Internship_Sehrish/FlyRank_ML_Internship_Sehrish


In [11]:
## One row = one page.
# Each page has a position_tier, content_type, and CTR, and I compute the opportunity score relative to its own group's expected CTR.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
visible = df[df["impressions_90d"] >= 100].copy()

group_stats = visible.groupby(["position_tier", "content_type"])["ctr"].agg(["mean", "count"])
group_stats = group_stats.rename(columns={"mean": "expected_ctr", "count": "group_n"})
visible = visible.merge(group_stats, on=["position_tier", "content_type"], how="left")
unit_df = visible[["position_tier", "content_type", "ctr", "expected_ctr", "group_n"]]
print(unit_df.head(10))


  position_tier     content_type   ctr  expected_ctr  group_n
0      striking  keyword article  0.76      0.255881     5752
1      page_3_5  keyword article  0.05      0.142711     5954
2      page_3_5  keyword article  0.09      0.142711     5954
3        page_1  keyword article  0.49      0.345805     8186
4      page_3_5  keyword article  0.13      0.142711     5954
5        page_1  keyword article  0.03      0.345805     8186
6      page_3_5  keyword article  0.06      0.142711     5954
7      page_3_5  keyword article  0.09      0.142711     5954
8        page_1  keyword article  0.16      0.345805     8186
9         top_3  keyword article  1.55      0.310417      527


In [12]:
## Note: even within well-sampled groups, individual rows can have implausible CTR values
# e.g. row 9 above: ctr=1.55 vs expected_ctr=0.31, n=527.
# A reminder that group-level sample size alone doesn't guarantee every row is clean, and any scoring
# method should also handle per-row outliers, not just group-level confidence.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [13]:
## A fixed rule would treat every group the same regardless of how much data backs that group's average.
# My Week 1 discovery already showed this breaks down: comparison article and feedly article
# have as few as 1-5 rows at top_3, so their "group average" is barely a real number.
# A fixed rule would flag or clear pages based on noise.

## An ML/statistical approach can weight the gap by group_n
# shrinking scores toward zero confidence when a group has too few pages to trust, and only
# surfacing high-confidence gaps for review.
# A single fixed threshold can't express that trade-off;
# it either ignores sample size entirely or requires hand-tuning
# A separate rule for every group size, which doesn't scale as new content_types or tiers appear.


## Self-check

Before you submit, confirm each line honestly:

- Every section above is filled — markdown thinking AND the code that backs it
- The notebook runs top to bottom with no errors (Runtime → Run all)
- No client names, URLs, or private queries anywhere
- My claims use careful words: observed, measured, directional, decision-support
- Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.


